In [0]:
# Create a Delta table from the uploaded CSV
df = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv("/Volumes/fraud_detection/transactions/raw_data/transactions_10k.csv")

df.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud_detection.transactions.raw_transactions")

print(f"Rows: {df.count()}")
display(df.limit(5))

In [0]:
# Cell 2 — Feature Engineering
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

df = spark.table("fraud_detection.transactions.raw_transactions")

# Cast columns to correct types
df = df.withColumn("timestamp", F.to_timestamp("timestamp")) \
       .withColumn("amount", F.col("amount").cast("double")) \
       .withColumn("is_fraud", F.col("is_fraud").cast("integer"))

# Time features
df = df.withColumn("hour_of_day", F.hour("timestamp")) \
       .withColumn("day_of_week", F.dayofweek("timestamp")) \
       .withColumn("is_weekend", (F.dayofweek("timestamp").isin([1,7])).cast("integer")) \
       .withColumn("is_night", ((F.hour("timestamp") < 6) | (F.hour("timestamp") >= 22)).cast("integer"))

# Amount features
df = df.withColumn("amount_log", F.log1p("amount")) \
       .withColumn("is_high_amount", (F.col("amount") > 500).cast("integer"))

# Window aggregations (rolling stats per card)
w_1h  = Window.partitionBy("card_id").orderBy(F.col("timestamp").cast("long")).rangeBetween(-3600, 0)
w_24h = Window.partitionBy("card_id").orderBy(F.col("timestamp").cast("long")).rangeBetween(-86400, 0)

df = df.withColumn("txn_count_1h",    F.count("*").over(w_1h)) \
       .withColumn("txn_amount_1h",   F.sum("amount").over(w_1h)) \
       .withColumn("txn_count_24h",   F.count("*").over(w_24h)) \
       .withColumn("txn_amount_24h",  F.sum("amount").over(w_24h)) \
       .withColumn("avg_amount_24h",  F.avg("amount").over(w_24h))

# Velocity: time since last transaction
w_lag = Window.partitionBy("card_id").orderBy("timestamp")
df = df.withColumn("prev_timestamp", F.lag("timestamp", 1).over(w_lag)) \
       .withColumn("time_since_last_txn_sec",
                   F.when(F.col("prev_timestamp").isNull(), 86400)
                    .otherwise(F.col("timestamp").cast("long") - F.col("prev_timestamp").cast("long")))

# Categorical encoding
df = df.withColumn("is_online", (F.col("entry_mode") == "online").cast("integer")) \
       .withColumn("is_international", (F.col("country") != "US").cast("integer"))

# Fill nulls
feature_cols = ["amount", "amount_log", "is_high_amount", "hour_of_day", "day_of_week",
                "is_weekend", "is_night", "txn_count_1h", "txn_amount_1h",
                "txn_count_24h", "txn_amount_24h", "avg_amount_24h",
                "time_since_last_txn_sec", "is_online", "is_international"]

df = df.fillna(0, subset=feature_cols)

# Save enriched table
df.write.format("delta").mode("overwrite") \
  .saveAsTable("fraud_detection.transactions.enriched_transactions")

print(f"Enriched rows: {df.count()}")
print(f"Fraud rate: {df.filter('is_fraud=1').count() / df.count():.2%}")

In [0]:
# Cell 3a — Install dependencies
%pip install xgboost scikit-learn
dbutils.library.restartPython()

In [0]:
# Cell 3 — Train XGBoost model with MLflow
import mlflow
import mlflow.xgboost
import xgboost as xgb
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

mlflow.set_experiment("/Users/madathingalkrishna@gmail.com/fraud-detection-model-training")

feature_cols = ["amount", "amount_log", "is_high_amount", "hour_of_day", "day_of_week",
                "is_weekend", "is_night", "txn_count_1h", "txn_amount_1h",
                "txn_count_24h", "txn_amount_24h", "avg_amount_24h",
                "time_since_last_txn_sec", "is_online", "is_international"]

pdf = spark.table("fraud_detection.transactions.enriched_transactions") \
           .select(feature_cols + ["is_fraud"]) \
           .toPandas()

# Temporal split — no data leakage
split = int(len(pdf) * 0.8)
X_train = pdf[feature_cols].iloc[:split]
y_train = pdf["is_fraud"].iloc[:split]
X_test  = pdf[feature_cols].iloc[split:]
y_test  = pdf["is_fraud"].iloc[split:]

fraud_ratio = int((y_train == 0).sum() / (y_train == 1).sum())

with mlflow.start_run(run_name="xgboost_v1"):
    params = {
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "scale_pos_weight": fraud_ratio,
        "eval_metric": "aucpr",
        "random_state": 42
    }
    mlflow.log_params(params)

    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_prob = model.predict_proba(X_test)[:, 1]

    # Find best threshold
    best_f1, best_thresh = 0, 0.5
    for t in np.arange(0.1, 0.9, 0.05):
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, t

    y_pred_final = (y_prob >= best_thresh).astype(int)
    metrics = {
        "precision": float(precision_score(y_test, y_pred_final)),
        "recall":    float(recall_score(y_test, y_pred_final)),
        "f1_score":  float(best_f1),
        "roc_auc":   float(roc_auc_score(y_test, y_prob)),
        "threshold": float(best_thresh)
    }
    mlflow.log_metrics(metrics)
    input_example = X_train.iloc[:5]
    mlflow.xgboost.log_model(
        model, 
        "model",
        input_example=input_example,
        registered_model_name="fraud_detector"
    )

    print("✅ Model trained and registered!")
    for k, v in metrics.items():
        print(f"   {k}: {v:.4f}")

In [0]:
# Cell 4 — Score all transactions and write fraud alerts
import mlflow
import pandas as pd
from pyspark.sql import functions as F

# Load the registered model as a Spark UDF
model_uri = "models:/workspace.default.fraud_detector/1"
predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri, result_type="double")

feature_cols = ["amount", "amount_log", "is_high_amount", "hour_of_day", "day_of_week",
                "is_weekend", "is_night", "txn_count_1h", "txn_amount_1h",
                "txn_count_24h", "txn_amount_24h", "avg_amount_24h",
                "time_since_last_txn_sec", "is_online", "is_international"]

# Load enriched transactions
scored_df = spark.table("fraud_detection.transactions.enriched_transactions")

# Apply model
scored_df = scored_df.withColumn("fraud_score", predict_udf(*feature_cols)) \
                     .withColumn("predicted_fraud", (F.col("fraud_score") >= 0.75).cast("integer")) \
                     .withColumn("alert_level",
                         F.when(F.col("fraud_score") >= 0.90, "HIGH")
                          .when(F.col("fraud_score") >= 0.75, "MEDIUM")
                          .otherwise("LOW"))

# Save scored transactions
scored_df.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud_detection.transactions.scored_transactions")

# Save high-priority alerts only
alerts_df = scored_df.filter("predicted_fraud = 1") \
    .select("transaction_id", "card_id", "merchant_id", "amount", 
            "timestamp", "merchant_category", "entry_mode", "country",
            "fraud_score", "alert_level", "is_fraud")

alerts_df.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud_detection.transactions.fraud_alerts")

# Summary
total = scored_df.count()
predicted_fraud = scored_df.filter("predicted_fraud = 1").count()
actual_fraud = scored_df.filter("is_fraud = 1").count()
high_alerts = alerts_df.filter("alert_level = 'HIGH'").count()

print(f"✅ Scoring complete!")
print(f"   Total transactions scored: {total:,}")
print(f"   Predicted fraud:           {predicted_fraud:,}")
print(f"   Actual fraud:              {actual_fraud:,}")
print(f"   HIGH priority alerts:      {high_alerts:,}")

In [0]:
# Cell 5 — Micro-batch refresh (pandas-based scoring to avoid schema enforcement issues)
import time
import mlflow
import mlflow.xgboost
import pandas as pd
import numpy as np
import random
from datetime import datetime, timezone
from pyspark.sql import functions as F

# Load model once outside the loop
model_uri = "models:/workspace.default.fraud_detector/1"
loaded_model = mlflow.xgboost.load_model(model_uri)

feature_cols = ["amount","amount_log","is_high_amount","hour_of_day","day_of_week",
                "is_weekend","is_night","txn_count_1h","txn_amount_1h",
                "txn_count_24h","txn_amount_24h","avg_amount_24h",
                "time_since_last_txn_sec","is_online","is_international"]

def run_batch(batch_num):
    print(f"\n{'='*50}")
    print(f"Batch {batch_num} — {datetime.now().strftime('%H:%M:%S')}")
    
    random.seed(batch_num * 42)
    np.random.seed(batch_num * 42)
    
    n = 200
    fraud_n = int(n * 0.075)
    categories = ["online_retail","jewelry","electronics","money_transfer",
                  "gambling","grocery","restaurant","gas_station"]
    countries  = ["US"]*70 + ["NG"]*10 + ["RU"]*5 + ["BR"]*5 + ["CN"]*5 + ["GB"]*5
    entries    = ["chip","swipe","online","contactless"]
    
    rows = []
    now = datetime.now(timezone.utc)
    for i in range(n):
        is_fraud = 1 if i < fraud_n else 0
        amount = round(random.uniform(0.5, 200), 2) if (is_fraud and random.random() < 0.5) \
                 else round(random.uniform(1, 9000), 2)
        rows.append({
            "transaction_id":    f"txn_batch{batch_num}_{i:04d}",
            "card_id":           f"card_{random.randint(0,500):016x}",
            "merchant_id":       f"merch_{random.randint(0,1000):08d}",
            "amount":            float(amount),
            "timestamp":         now.isoformat(),
            "merchant_category": random.choice(categories),
            "entry_mode":        "online" if is_fraud else random.choice(entries),
            "country":           random.choice(["NG","RU","BR"]) if is_fraud else random.choice(countries),
            "is_fraud":          int(is_fraud),
            "device_id":         f"dev_{random.randint(0,10000):08d}",
            "ip_address":        f"{random.randint(1,255)}.{random.randint(0,255)}.0.1",
            "currency":          "USD",
            "channel":           "online" if is_fraud else "pos",
        })
    
    pdf = pd.DataFrame(rows)
    
    # Feature engineering in pandas
    pdf["hour_of_day"]   = pd.to_datetime(pdf["timestamp"]).dt.hour
    pdf["day_of_week"]   = pd.to_datetime(pdf["timestamp"]).dt.dayofweek
    pdf["is_weekend"]    = pdf["day_of_week"].isin([5,6]).astype(int)
    pdf["is_night"]      = ((pdf["hour_of_day"] < 6) | (pdf["hour_of_day"] >= 22)).astype(int)
    pdf["amount_log"]    = np.log1p(pdf["amount"])
    pdf["is_high_amount"]= (pdf["amount"] > 500).astype(int)
    pdf["txn_count_1h"]  = 1
    pdf["txn_amount_1h"] = pdf["amount"]
    pdf["txn_count_24h"] = 1
    pdf["txn_amount_24h"]= pdf["amount"]
    pdf["avg_amount_24h"]= pdf["amount"]
    pdf["time_since_last_txn_sec"] = 300.0
    pdf["is_online"]     = (pdf["entry_mode"] == "online").astype(int)
    pdf["is_international"] = (pdf["country"] != "US").astype(int)
    
    # Score with model directly (no Spark UDF, no schema enforcement)
    X = pdf[feature_cols].astype(float)
    pdf["fraud_score"]    = loaded_model.predict_proba(X)[:, 1]
    pdf["predicted_fraud"]= (pdf["fraud_score"] >= 0.75).astype(int)
    pdf["alert_level"]    = pd.cut(
        pdf["fraud_score"],
        bins=[-0.001, 0.749, 0.899, 1.0],
        labels=["LOW","MEDIUM","HIGH"]
    ).astype(str)
    
    # Write to Delta via Spark
    scored_sdf = spark.createDataFrame(pdf)
    
    scored_sdf.write.format("delta").mode("append").option("mergeSchema","true") \
        .saveAsTable("fraud_detection.transactions.scored_transactions")
    
    alerts_sdf = scored_sdf.filter("predicted_fraud = 1") \
        .select("transaction_id","card_id","merchant_id","amount",
                "timestamp","merchant_category","entry_mode","country",
                "fraud_score","alert_level","is_fraud")
    
    alerts_sdf.write.format("delta").mode("append").option("mergeSchema","true") \
        .saveAsTable("fraud_detection.transactions.fraud_alerts")
    
    pred_fraud = int(pdf["predicted_fraud"].sum())
    print(f"   New transactions: {n}")
    print(f"   Predicted fraud:  {pred_fraud}")
    print(f"   Tables updated ✅")

# Run 5 batches
for i in range(1, 6):
    run_batch(i)
    if i < 5:
        print("   Sleeping 60s...")
        time.sleep(60)

print("\n✅ Micro-batch pipeline complete! Refresh your dashboard to see updated totals.")